Create data from a CoxPH model fitted on ground truth concepts labels

In [ ]:
import pandas as pd
from sksurv.linear_model import CoxPHSurvivalAnalysis
# from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.util import Surv
import numpy as np
from matplotlib import pyplot as plt
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import KFold

In [ ]:
df = pd.read_csv("brca.csv")

concepts = ["Stage", "Age", "RNA_Bio_ter"]
concept_states = [4, 3, 3]

for n,concept in enumerate(concepts):
    for state in range(concept_states[n]):
        df[f'{concept}_class_{state}'] = np.where(df[concept] == state, 1, 0)

df['noise'] = np.random.randn(len(df))

In [ ]:
def create_folds(df,K):

    '''split data into K folds'''

    kf = KFold(
    n_splits=5
    ,shuffle=True        
    )

    folds = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        folds.append((train_df, val_df))
        
    return folds


In [ ]:
def fit_coxPH(train_df,feature_list):

    y = Surv.from_dataframe("event", "time", train_df)

    X = train_df[feature_list].values

    model = CoxPHSurvivalAnalysis()
    model.fit(X, y)
    return model


In [ ]:
def get_c_index(model,test_df,feature_list):


    '''calculate the C-Index for a given model'''

    X = test_df[feature_list].values
    
    hazards = model.predict(X)
    
    y = Surv.from_dataframe("event", "time", test_df)

    c_index, _, _, _, _ = concordance_index_censored(
        y["event"],
        y["time"],
        hazards
    )

    return c_index

In [ ]:
# need to drop one from each to make it identifiable
features = [
#  'Stage_class_0'
 'Stage_class_1'
, 'Stage_class_2'
, 'Stage_class_3'
# , 'Age_class_0'
, 'Age_class_1'
, 'Age_class_2'
# , 'RNA_Bio_ter_class_0'
, 'RNA_Bio_ter_class_1'
, 'RNA_Bio_ter_class_2'
 ]

In [ ]:
# the feature sets we want to test
features_sets = {

    'noise': ['noise']

    ,"Stage": ['Stage_class_1', 'Stage_class_2', 'Stage_class_3']
    ,"Age": ['Age_class_1', 'Age_class_2']
    ,"RNA": ['RNA_Bio_ter_class_1', 'RNA_Bio_ter_class_2']

    ,"Stage + Age": ['Stage_class_1', 'Stage_class_2', 'Stage_class_3', 'RNA_Bio_ter_class_1', 'RNA_Bio_ter_class_2']
    ,"Stage + RNA": ['Stage_class_1', 'Stage_class_2', 'Stage_class_3', 'Age_class_1', 'Age_class_2']
    ,"Age + RNA": ['Age_class_1', 'Age_class_2', 'RNA_Bio_ter_class_1', 'RNA_Bio_ter_class_2']

    ,"All": features


}



In [ ]:
# for each feature set, run n_iter 5 folds and log results
n_iter = 2
run_results = [] # to store results

for set in features_sets.keys():
    

    for k in range(n_iter):

        folds = create_folds(df,5)


        for n,fold in enumerate(folds):
        
            train_df, test_df = fold[0], fold[1]
            
            model = fit_coxPH(fold[0],features_sets[set])

            c_index = get_c_index(model,test_df,features_sets[set])
            # (set,run,fold,c_index)
            run_results.append((set,f'run_{k}',f'fold_{n}',c_index.item()))
run_results

/Users/edwarddaniel/Desktop/CBM/.venv/lib/python3.13/site-packages/sksurv/linear_model/coxph.py:197: RuntimeWarning: overflow encountered in exp
  risk_set2 += np.exp(xw[k])
/Users/edwarddaniel/Desktop/CBM/.venv/lib/python3.13/site-packages/sksurv/linear_model/coxph.py:200: RuntimeWarning: overflow encountered in exp
  risk_set += np.exp(xw[k])
/Users/edwarddaniel/Desktop/CBM/.venv/lib/python3.13/site-packages/sksurv/linear_model/coxph.py:197: RuntimeWarning: overflow encountered in exp
  risk_set2 += np.exp(xw[k])
/Users/edwarddaniel/Desktop/CBM/.venv/lib/python3.13/site-packages/sksurv/linear_model/coxph.py:200: RuntimeWarning: overflow encountered in exp
  risk_set += np.exp(xw[k])
/Users/edwarddaniel/Desktop/CBM/.venv/lib/python3.13/site-packages/sksurv/linear_model/coxph.py:197: RuntimeWarning: overflow encountered in exp
  risk_set2 += np.exp(xw[k])
/Users/edwarddaniel/Desktop/CBM/.venv/lib/python3.13/site-packages/sksurv/linear_model/coxph.py:200: RuntimeWarning: overflow encoun

[('noise', 'run_0', 'fold_0', 0.4233661075766339),
 ('noise', 'run_0', 'fold_1', 0.51640625),
 ('noise', 'run_0', 'fold_2', 0.3341487279843444),
 ('noise', 'run_0', 'fold_3', 0.512285581826611),
 ('noise', 'run_0', 'fold_4', 0.5421146953405018),
 ('noise', 'run_1', 'fold_0', 0.5324939073923639),
 ('noise', 'run_1', 'fold_1', 0.6378633150039277),
 ('noise', 'run_1', 'fold_2', 0.518979057591623),
 ('noise', 'run_1', 'fold_3', 0.5195557701593433),
 ('noise', 'run_1', 'fold_4', 0.5236920039486673),
 ('Stage', 'run_0', 'fold_0', 0.7994530537830447),
 ('Stage', 'run_0', 'fold_1', 0.6236717827626919),
 ('Stage', 'run_0', 'fold_2', 0.7062466343564889),
 ('Stage', 'run_0', 'fold_3', 0.654490106544901),
 ('Stage', 'run_0', 'fold_4', 0.7038461538461539),
 ('Stage', 'run_1', 'fold_0', 0.6087570621468926),
 ('Stage', 'run_1', 'fold_1', 0.737037037037037),
 ('Stage', 'run_1', 'fold_2', 0.7787418655097614),
 ('Stage', 'run_1', 'fold_3', 0.7066820276497696),
 ('Stage', 'run_1', 'fold_4', 0.66394779771

In [ ]:
# save results to a csv
results_df = pd.DataFrame(run_results, columns=['xp_id', 'run', 'fold', 'task_c_index'])
results_df.to_csv('cox_results.csv',index=False)